In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os, pickle, gc

import lightgbm as lgb
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
plt.rcParams['font.family'] = 'NanumGothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False



In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 함수 생성

In [4]:
def load_file(prep_fold):
  fn = os.listdir(path+prep_fold)

  for f in fn :
    if 'test' in f:
      test_n = f
    else :
      train_n = f

  train = pd.read_parquet(path + prep_fold +'/' + train_n)
  test =  pd.read_parquet(path + prep_fold +'/' + test_n)

  print(f'train file name : {train_n}')
  print(f'test file name : {test_n}')
  return train, test

In [5]:
def cols_prep(train_df, test_df) :
  # train columns drop
  drop_cols = ['기준년월','ID','Segment']
  x = train_df.drop(columns=drop_cols)
  y = train_df['Segment']

  # test 데이터 컬럼 train 데이터와 동일하게 맞추기
  X_test = test_df[x.columns]

  return x, y, X_test

In [6]:
def modeling(learn_rate, version) :
  model = lgb.LGBMClassifier(
      objective='multiclass',
      num_class=5,
      n_estimators=2000,
      learning_rate=learn_rate,
      random_state=42,
      n_jobs=-1,
      min_gain_to_split=1e-5,
      min_data_in_leaf=10,
      subsample=0.8,
      colsample_bytree=0.8
  )

  model.fit(
      x_train, y_train,
      eval_set=[(x_test, y_test)],
      eval_metric='multi_logloss',
      callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=100)]
  )

  # 모델 저장
  pickle.dump(model, open(f'/content/drive/MyDrive/Colab_Notebooks/LGBM_{version}_learning_rate{learn_rate}.pkl', 'wb'))

  return model

In [7]:
def evaluate_multiclass(y_true, y_pred, y_proba, learn_rate, version):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')

    # ROC AUC: 다중 분류는 one-vs-rest 방식 필요
    try:
        roc = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')
    except:
        roc = np.nan  # 예외 발생 시

    # G-Mean: 각 클래스의 recall 평균으로 근사
    cm = confusion_matrix(y_true, y_pred)
    recalls = []
    for i in range(len(cm)):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        if tp + fn != 0:
            recalls.append(tp / (tp + fn))
    gmean = np.sqrt(np.prod(recalls)) if all(r > 0 for r in recalls) else 0

    score_dict = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-score': f1,
        'ROC AUC': roc,
        'G-Mean': gmean
    }

    pd.DataFrame([score_dict]).to_csv(f'/content/drive/MyDrive/Colab_Notebooks/LGBM_{version}_score_learn_rate{learn_rate}.csv', index=False)

    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-score  : {f1:.4f}")
    print(f"ROC AUC   : {roc:.4f}")
    print(f"G-Mean    : {gmean:.4f}")

In [8]:
def save_data(test, final_pred, learn_rate, version):
  res_df = pd.DataFrame({
      'ID': test.ID,
      'Segment': final_pred
  })

  res_df.to_csv(f'/content/drive/MyDrive/Colab_Notebooks/submission_raw_{version}_learning_rate_{learn_rate}.csv', index=False)

  submission_unique = res_df.groupby('ID')['Segment'].agg(lambda x: x.mode().iloc[0]).reset_index()
  submission_unique.to_csv(f'/content/drive/MyDrive/Colab_Notebooks/submission_{version}_learning_rate_{learn_rate}.csv', index=False)

- 데이터 경로 확인 및 변수 할당

In [9]:
# 데이터 경로
print(os.getcwd())
path = '/content/drive/MyDrive/파이널 프로젝트/2025_07_09/' #'/content/drive/MyDrive/Colab_Notebooks/01.like_lion_final_prj/'
prep_folder = os.listdir(path)

/content


In [10]:
prep_folder

['분산(0.01),열의상관(0.7이상),Segment의상관(0.1이상)',
 '분산(0.01),열의상관(0.8이상),Segment의상관(0.1이상)',
 '분산(0.01),열의상관(0.8이상),Segment의상관(0.1이상,열의 상관중 Segment와 0.3이상 상관이 있으면 열삭제 x)']

In [11]:
print(os.listdir(path+prep_folder[0]))

['Segment_merge_ver_02.parquet', 'Segment_merge_test_ver_02.parquet']


### 전처리 ver02
- 파일 로드

In [12]:
train, test = load_file(prep_folder[0])

train file name : Segment_merge_ver_02.parquet
test file name : Segment_merge_test_ver_02.parquet


In [13]:
print(train.shape)
print(test.shape)

(2400000, 66)
(600000, 65)


In [14]:
x, y, X_test = cols_prep(train, test)

In [15]:
  # 데이터 분할
  x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

- 모델링
  - learning rate = 0.1

In [16]:
# 모델링
## learning rate 0.1
model = modeling(0.1, 'ver02')

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.172123 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8996
[LightGBM] [Info] Number of data points in the train set: 1920000, number of used features: 63
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] mi

In [17]:
# 모델 평가 지표
# y_test: 실제값 (0, 1, 2 등)
# model: LightGBM 또는 다른 모델
y_pred = model.predict(x_test)
y_proba = model.predict_proba(x_test)  # shape: (n_samples, n_classes)

evaluate_multiclass(y_test, y_pred, y_proba, 0.1, 'ver02')

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
Accuracy  : 0.8786
Precision : 0.4853
Recall    : 0.4438
F1-score  : 0.4602
ROC AUC   : 0.7285
G-Mean    : 0.0323


- 결과 예측

In [18]:
# 5. 예측 수행
final_pred = model.predict(X_test)               # 예측 레이블
final_proba = model.predict_proba(X_test)

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05


In [19]:
# 6. 결과 저장
save_data(test, final_pred, 0.1, 'ver02')

NameError: name 'model' is not defined

- 모델링
  - learning rate = 0.05

In [25]:
# 모델링
## learning rate 0.05
model = modeling(0.05, 'ver02')

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.168281 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8996
[LightGBM] [Info] Number of data points in the train set: 1920000, number of used features: 63
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] mi

In [26]:
# 모델 평가 지표
y_pred = model.predict(x_test)
y_proba = model.predict_proba(x_test)  # shape: (n_samples, n_classes)

evaluate_multiclass(y_test, y_pred, y_proba, 0.05, 'ver02')

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
Accuracy  : 0.8809
Precision : 0.5099
Recall    : 0.5033
F1-score  : 0.4993
ROC AUC   : 0.7761
G-Mean    : 0.0966


In [27]:
# 5. 예측 수행
final_pred = model.predict(X_test)               # 예측 레이블
final_proba = model.predict_proba(X_test)

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05


In [28]:
# 6. 결과 저장
save_data(test, final_pred, 0.05, 'ver02')

## 전처리 ver 03
- 파일 로드

In [29]:
prep_folder[1]

'분산(0.01),열의상관(0.8이상),Segment의상관(0.1이상)'

In [30]:
train, test = load_file(prep_folder[1])

train file name : Segment_merge_ver_03.parquet
test file name : Segment_merge_test_ver_03.parquet


In [31]:
print(train.shape)
print(test.shape)

(2400000, 66)
(600000, 65)


In [32]:
x, y, X_test = cols_prep(train, test)

In [33]:
  # 데이터 분할
  x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

- 모델링
  - learning rate = 0.1

In [34]:
# 모델링
## learning rate 0.1
model = modeling(0.1, 'ver03')

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.173563 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8996
[LightGBM] [Info] Number of data points in the train set: 1920000, number of used features: 63
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] mi

In [35]:
# 모델 평가 지표
y_pred = model.predict(x_test)
y_proba = model.predict_proba(x_test)  # shape: (n_samples, n_classes)

evaluate_multiclass(y_test, y_pred, y_proba, 0.1, 'ver03')

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
Accuracy  : 0.8786
Precision : 0.4853
Recall    : 0.4438
F1-score  : 0.4602
ROC AUC   : 0.7285
G-Mean    : 0.0323


- 결과 예측

In [36]:
# 5. 예측 수행
final_pred = model.predict(X_test)               # 예측 레이블
final_proba = model.predict_proba(X_test)

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05


In [37]:
# 6. 결과 저장
save_data(test, final_pred, 0.1, 'ver03')

- 모델링
  - learning rate = 0.05

In [38]:
# 모델링
## learning rate 0.05
model = modeling(0.05, 'ver03')

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.168894 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8996
[LightGBM] [Info] Number of data points in the train set: 1920000, number of used features: 63
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] mi

In [39]:
# 모델 평가 지표
y_pred = model.predict(x_test)
y_proba = model.predict_proba(x_test)  # shape: (n_samples, n_classes)

evaluate_multiclass(y_test, y_pred, y_proba, 0.05, 'ver03')

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
Accuracy  : 0.8809
Precision : 0.5099
Recall    : 0.5033
F1-score  : 0.4993
ROC AUC   : 0.7761
G-Mean    : 0.0966


In [40]:
# 5. 예측 수행
final_pred = model.predict(X_test)               # 예측 레이블
final_proba = model.predict_proba(X_test)

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05


In [41]:
# 6. 결과 저장
save_data(test, final_pred, 0.05, 'ver03')

NameError: name 'res_df' is not defined

## 전처리 ver05
- 파일 로드

In [43]:
prep_folder[2]

'분산(0.01),열의상관(0.8이상),Segment의상관(0.1이상,열의 상관중 Segment와 0.3이상 상관이 있으면 열삭제 x)'

In [44]:
train, test = load_file(prep_folder[2])

train file name : Segment_merge_ver_05.parquet
test file name : Segment_merge_test_ver_05.parquet


In [45]:
print(train.shape)
print(test.shape)

(2400000, 138)
(600000, 137)


In [46]:
x, y, X_test = cols_prep(train, test)

In [47]:
  # 데이터 분할
  x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

- 모델링
  - learning rate 0.1

In [48]:
# 모델링
## learning rate 0.1
model = modeling(0.1, 'ver05')

# 모델 평가 지표
y_pred = model.predict(x_test)
y_proba = model.predict_proba(x_test)  # shape: (n_samples, n_classes)

evaluate_multiclass(y_test, y_pred, y_proba, 0.1, 'ver05')

# 6. 결과 저장
save_data(test, final_pred, 0.1, 'ver05')

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.181759 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 24919
[LightGBM] [Info] Number of data points in the train set: 1920000, number of used features: 135
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignor

  - learning rate = 0.05

In [49]:
# 모델링
## learning rate 0.05
model = modeling(0.05, 'ver05')

# 모델 평가 지표
y_pred = model.predict(x_test)
y_proba = model.predict_proba(x_test)  # shape: (n_samples, n_classes)

evaluate_multiclass(y_test, y_pred, y_proba, 0.05, 'ver05')

# 6. 결과 저장
save_data(test, final_pred, 0.05, 'ver05')

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_gain_to_split is set=1e-05, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=1e-05
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.834853 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 24919
[LightGBM] [Info] Number of data points in the train set: 1920000, number of used features: 135
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] 

In [50]:
del y_pred, y_proba